# Vector DB

In [1]:
from dotenv import load_dotenv
import os

# langchain + ChromaDB
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

c:\Users\sb730\anaconda3\envs\deepl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_docs(sentences_list: list[str]) -> list[Document]:
  '''
  문자열 문장을 랭체인 Document 객체로 변환
  각 문서의 메타 데이터 자동 추가
  - id : 문서번호
  - source : 문서 출처
  '''

  documents = []

  for idx, sentence in enumerate(sentences_list, start = 1):
    # 공백만 있는 데이터는 생략
    if not sentence.strip():
      continue

    document = Document(
      page_content=sentence.strip(),
      metadata = {
        "id" : idx,
        "source" : "loc_example"
      }
    )
    documents.append(document)

  return documents

In [3]:
sentences = [
    "고양이가 창가에서 햇볕을 쬐고 있다",
    "새끼 고양이가 실타래를 가지고 논다",
    "강아지가 공원에서 뛰어다닌다",
    "강아지가 주인을 보고 꼬리를 흔든다",
    "호랑이가 숲속을 걸어 다닌다",
    "사자가 초원에서 사냥을 준비한다",
    "코끼리가 긴 코로 나뭇가지를 잡는다",
    "기린이 나무 위 잎을 먹는다",
    "원숭이가 바나나를 먹는다",
    "펭귄이 얼음 위에서 걸어간다",
    "사과가 빨갛게 익었다",
    "바나나가 노랗게 잘 익어 달콤하다",
    "포도가 달콤하고 신선하다",
    "빵이 오븐에서 갓 구워졌다",
    "라면이 뜨겁게 끓고 있다",
    "김치찌개에서 얼큰한 냄새가 난다",
    "초밥이 접시에 예쁘게 놓여 있다",
    "피자가 치즈로 가득 덮여 있다",
    "햄버거에 고기 패티가 두껍다",
    "커피 향이 방 안에 가득 퍼졌다",
    "서울의 남산타워가 빛나고 있다",
    "한강 다리 위에서 야경을 본다",
    "제주도의 바닷가가 파랗게 빛난다",
    "부산 해운대 해수욕장에 사람이 많다",
    "경복궁의 전통 건물이 웅장하다",
    "뉴욕의 자유의 여신상이 우뚝 서 있다",
    "파리의 에펠탑이 밤에 반짝인다",
    "런던의 빅벤 시계탑이 울린다",
    "도쿄의 시부야 거리가 붐빈다",
    "로마의 콜로세움이 고대의 흔적을 보여준다"
]

In [4]:
documents = create_docs(sentences)

for doc in documents:
  print(f"내용 : {doc.page_content}")
  print(f"메타 데이터 : {doc.metadata}")
  print('='*10)

내용 : 고양이가 창가에서 햇볕을 쬐고 있다
메타 데이터 : {'id': 1, 'source': 'loc_example'}
내용 : 새끼 고양이가 실타래를 가지고 논다
메타 데이터 : {'id': 2, 'source': 'loc_example'}
내용 : 강아지가 공원에서 뛰어다닌다
메타 데이터 : {'id': 3, 'source': 'loc_example'}
내용 : 강아지가 주인을 보고 꼬리를 흔든다
메타 데이터 : {'id': 4, 'source': 'loc_example'}
내용 : 호랑이가 숲속을 걸어 다닌다
메타 데이터 : {'id': 5, 'source': 'loc_example'}
내용 : 사자가 초원에서 사냥을 준비한다
메타 데이터 : {'id': 6, 'source': 'loc_example'}
내용 : 코끼리가 긴 코로 나뭇가지를 잡는다
메타 데이터 : {'id': 7, 'source': 'loc_example'}
내용 : 기린이 나무 위 잎을 먹는다
메타 데이터 : {'id': 8, 'source': 'loc_example'}
내용 : 원숭이가 바나나를 먹는다
메타 데이터 : {'id': 9, 'source': 'loc_example'}
내용 : 펭귄이 얼음 위에서 걸어간다
메타 데이터 : {'id': 10, 'source': 'loc_example'}
내용 : 사과가 빨갛게 익었다
메타 데이터 : {'id': 11, 'source': 'loc_example'}
내용 : 바나나가 노랗게 잘 익어 달콤하다
메타 데이터 : {'id': 12, 'source': 'loc_example'}
내용 : 포도가 달콤하고 신선하다
메타 데이터 : {'id': 13, 'source': 'loc_example'}
내용 : 빵이 오븐에서 갓 구워졌다
메타 데이터 : {'id': 14, 'source': 'loc_example'}
내용 : 라면이 뜨겁게 끓고 있다
메타 데이터 : {'id': 15, 'source': 'loc_example'}
내용 : 김치찌개

In [5]:
import shutil

embedding = OpenAIEmbeddings(
  model = "text-embedding-3-large"
  
)

db_path = '../chroma_db'

if os.path.exists(db_path):
  # os.rmdir(db_path)
  shutil.rmtree(db_path)

vector_db = Chroma(
  collection_name="loc_docs",
  embedding_function=embedding,
  persist_directory=db_path,

  # 벡터간 거리 유사도를 cos 유사도를 사용함
  collection_metadata={
    "hnsw:space" : "cosine"
  }
)

In [6]:
# 고유 ID 생성
documents_ids = [f'doc_{doc.metadata['id']}' for doc in documents]

# 추가
saved_ids = vector_db.add_documents(
  documents=documents,
  ids = documents_ids
)

len(saved_ids)

30

In [8]:
query = input('검색 문장을 입력하세요').strip()
print(f"query = {query}")

res = vector_db._similarity_search_with_relevance_scores(
  query = query,
  k = 3
)
res

query = 강아지가뛰어다닌다


[(Document(id='doc_3', metadata={'source': 'loc_example', 'id': 3}, page_content='강아지가 공원에서 뛰어다닌다'),
  0.8063347935676575),
 (Document(id='doc_4', metadata={'source': 'loc_example', 'id': 4}, page_content='강아지가 주인을 보고 꼬리를 흔든다'),
  0.5439070463180542),
 (Document(id='doc_5', metadata={'source': 'loc_example', 'id': 5}, page_content='호랑이가 숲속을 걸어 다닌다'),
  0.5158011317253113)]

In [11]:
for doc, score in res:
  print(f"내용 : {doc.page_content}")
  print(f"메타데이터 : {doc.metadata}")
  print(f"점수 : {score}")
  print("===================================")

내용 : 강아지가 공원에서 뛰어다닌다
메타데이터 : {'source': 'loc_example', 'id': 3}
점수 : 0.8063347935676575
내용 : 강아지가 주인을 보고 꼬리를 흔든다
메타데이터 : {'source': 'loc_example', 'id': 4}
점수 : 0.5439070463180542
내용 : 호랑이가 숲속을 걸어 다닌다
메타데이터 : {'source': 'loc_example', 'id': 5}
점수 : 0.5158011317253113


In [14]:
result = vector_db.get(
    include=["embeddings", "documents", "metadatas"]
)

print("총 개수:", len(result["ids"]))
print("첫번째 id:", result["ids"][0])
print("첫번째 문장:", result["documents"][0])
print("첫번째 metadata:", result["metadatas"][0])
print("첫번째 벡터 차원:", len(result["embeddings"][0]))
print("첫번째 벡터 앞부분:", result["embeddings"][0][:-10])  # 앞 5개 숫자만 미리보기

총 개수: 30
첫번째 id: doc_1
첫번째 문장: 고양이가 창가에서 햇볕을 쬐고 있다
첫번째 metadata: {'id': 1, 'source': 'loc_example'}
첫번째 벡터 차원: 3072
첫번째 벡터 앞부분: [ 0.01013947  0.00320625  0.00322914 ...  0.00362015 -0.00511551
  0.00231171]
